# 02 — Klasik Baseline: blob detection + Hungarian linking

Detection = **Otsu eşiği → çekirdek-boyutlu local-max → bağlı bileşen ağırlık merkezi**
(`center_of_mass`). Linking = **Hungarian, 8 µm kapılı**. Sıfır ek bağımlılık.

> **Not:** Bu hat, Erdem'in `erd_exp/eda_detailed` baseline'ından uyarlandı (yerelde
> ~0.78 micro edge-Jaccard, LB ~0.7). Bizim önceki `top-N + adaptif hedef` yaklaşımımız
> tek çekirdeği birden çok kez tespit edip (kare-içi komşu ~5 µm) linking'i kırıyordu;
> `center_of_mass` bunu tek merkeze indirgiyor. Ayrıntı: `docs/EDA_BULGULARI.md`.

> ### ⚠️ SUBMIT KURALLARI
> - **Internet KAPALI** (Settings → Internet = Off). `pip install` YOK.
> - **`erdeemt/cell-tracking-libs` dataset'i EKLİ** — `zarr` imajda yok.
> - **Code competition:** notebook gizli test setiyle yeniden çalışır → test isimleri
>   **hardcode edilmez**, `test/*.zarr` dinamik gezilir.
> - Çıktı: **`/kaggle/working/submission.csv`**.

### Metriğin şekli (neden recall'a odaklanıyoruz)
GT seyrek → bir tahmin kenarının **iki ucu da** GT node'a eşleşmezse kenar **yok sayılır**
(FP≈0). Yani `adjusted_jaccard ≈ (GT kenar recall) × ceza`. Ceza zayıf (`[0.9, 1.1]`),
fazla-tahmin neredeyse bedava. **Oyun: her GT kenarının iki ucuna tespit koy + bağla.**

## 0 — Kurulum (zarr utility dataset'ten)

In [ ]:
# zarr bu imajda YOK -> erdeemt/cell-tracking-libs dataset'inden sys.path ile.
# append! insert(0) DEGIL: pylibs'te numpy 2.5.1 var, ortamin 2.0.2'sini golgelememeli
# (scipy/skimage 2.0.2'ye karsi derlenmis).
import sys, os
from pathlib import Path

LIBS = Path('/kaggle/input/datasets/erdeemt/cell-tracking-libs/pylibs')
if not LIBS.exists():
    hits = [Path(r) for r, d, f in os.walk('/kaggle/input') if os.path.basename(r) == 'pylibs']
    assert hits, 'cell-tracking-libs dataset i notebook a ekli degil!'
    LIBS = hits[0]
sys.path.append(str(LIBS))

import time, warnings
import numpy as np
import pandas as pd
import zarr
from scipy import ndimage as ndi
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from skimage.filters import threshold_otsu
warnings.filterwarnings('ignore')

print('zarr ', zarr.__version__, '| numpy', np.__version__)
assert np.__version__.startswith('2.0'), f'numpy golgelendi: {np.__version__} — insert(0) mi kullandin?'
print('hazir')

## 1 — Yapılandırma & veri kökü (isim hardcode YOK)

In [ ]:
SCALE_ZYX = (1.625, 0.40625, 0.40625)   # um/piksel (Z,Y,X) — attrs'ten dogrulandi
S = np.array(SCALE_ZYX, dtype=np.float32)

LINK_MAX_UM = 8.0      # linking arama yaricapi (EDA: hareket 99p ~8 um)
MATCH_UM    = 7.0      # metrik eslesme toleransi
SIGMA = (1, 2, 2)      # Gauss yumusatma (voxel; Z ekseni ince oldugu icin 1)
FOOT  = (3, 11, 11)    # local-max footprint ~ cekirdek boyutu (~10 um)

OUT_CSV = Path('/kaggle/working/submission.csv')
INPUT = Path('/kaggle/input')

def find_root():
    """train/ + test/ iceren yarisma dizinini bul (mount yolu degisebilir)."""
    st = [(INPUT, 0)]
    while st:
        b, d = st.pop()
        try:
            if (b / 'train').is_dir() and (b / 'test').is_dir():
                return b
        except Exception:
            pass
        if d < 4:
            for c in sorted(b.iterdir()):
                if c.is_dir() and not c.name.endswith(('.zarr', '.geff')):
                    st.append((c, d + 1))
    raise RuntimeError('yarisma koku bulunamadi')

ROOT = find_root(); TRAIN = ROOT / 'train'; TEST = ROOT / 'test'
test_names = sorted(p.stem for p in TEST.glob('*.zarr'))
print('ROOT:', ROOT)
print(f'train={len(list(TRAIN.glob("*.zarr")))} | test={len(test_names)}')
print('test:', test_names[:10], '...' if len(test_names) > 10 else '')

# DEV mi GERCEK RERUN mu? Placeholder test = train'den kopya -> GT'leri train'de.
# Gercek rerun'da gizli test isimleri train'de YOK -> tum dogrulama/eval atlanir.
DEV = len(test_names) > 0 and (TRAIN / (test_names[0] + '.geff')).exists()
print('\nMOD:', 'DEV (placeholder test, GT var)' if DEV else 'GERCEK RERUN (gizli test, GT yok)')

## 2 — Okuyucular

In [ ]:
def open_image(zpath):
    """OME-Zarr goruntu dizisini ac -> (T,Z,Y,X)."""
    n = zarr.open(str(zpath), mode='r')
    a = dict(n.attrs)
    ms = a.get('multiscales') or (a['ome'].get('multiscales') if isinstance(a.get('ome'), dict) else None)
    if ms:
        return n[ms[0]['datasets'][0]['path']]
    return n['0'] if '0' in list(n.keys()) else n

def load_geff(gp):
    """GEFF -> (nodes_df[id,t,z,y,x], edges (E,2)). Duz zarr; geff/tracksdata gerekmiyor."""
    g = zarr.open(str(gp), mode='r')
    ids = np.asarray(g['nodes/ids'])
    d = {'id': ids}
    for k in ('t', 'z', 'y', 'x'):
        d[k] = np.asarray(g[f'nodes/props/{k}/values'])
    return pd.DataFrame(d), np.asarray(g['edges/ids'])

print('ok')

## 3 — Detection: Otsu → local-max → bağlı bileşen ağırlık merkezi

`(sm==maximum_filter(sm)) & (sm>otsu)` peak maskesini `ndi.label` ile bağlı bileşenlere
ayırıp her bileşenin `center_of_mass`'ını alıyoruz — **çekirdek başına tek, alt-voxel merkez**.
Bu, ham peak voxel'i kullanmaya kıyasla çift-tespiti önler (linking'i korur).

In [ ]:
def detect_frame(v):
    sm = ndi.gaussian_filter(v.astype(np.float32), sigma=SIGMA)
    thr = threshold_otsu(sm)
    mx = ndi.maximum_filter(sm, size=FOOT)
    peaks = (sm == mx) & (sm > thr)
    lbl, n = ndi.label(peaks)
    if n == 0:
        return np.zeros((0, 3), np.float32)
    c = ndi.center_of_mass(sm, lbl, np.arange(1, n + 1))
    return np.asarray(c, dtype=np.float32)     # (N,3) voxel (z,y,x)


# hiz + yogunluk kontrolu (T'yi HARDCODE ETME)
_arr = open_image(TEST / (test_names[0] + '.zarr'))
_T = _arr.shape[0]
t0 = time.time(); _c = detect_frame(np.asarray(_arr[_T // 2])); dt = time.time() - t0
print(f'shape {_arr.shape} | 1 kare {dt:.2f}s | cekirdek {len(_c)}')
print(f'tahmini 1 dataset ({_T} kare): {dt*_T:.0f}s | {len(test_names)} test: {dt*_T*len(test_names)/60:.1f} dk')

## 4 — Linking: Hungarian, 8 µm kapılı

Ardışık karelerde optimal 1-1 eşleştirme; 8 µm üstü yasak. Eşleşmeyen = beliriş/kayboluş.
Gap-closing yok (EDA: soy içinde zamansal boşluk yok).

In [ ]:
def link_pairs(A, B):
    if len(A) == 0 or len(B) == 0:
        return []
    D = cdist(A * S, B * S)                        # um mesafe
    cost = np.where(D <= LINK_MAX_UM, D, 1e6)      # kapi
    r, c = linear_sum_assignment(cost)
    return [(int(i), int(j)) for i, j in zip(r, c) if D[i, j] <= LINK_MAX_UM]


def track_dataset(arr):
    """Bir .zarr -> (nodes, edges). node_id dataset icinde 1'den ardisik."""
    T = arr.shape[0]
    cents = [detect_frame(np.asarray(arr[t])) for t in range(T)]
    nodes, edges, off, nid = [], [], [], 1
    for t, c in enumerate(cents):
        off.append(nid)
        for p in c:
            # koordinatlar TAMSAYI voxel (7 um tolerans yaninda yuvarlama ihmal edilebilir)
            nodes.append((nid, t, int(round(p[0])), int(round(p[1])), int(round(p[2])))); nid += 1
    for t in range(T - 1):
        for i, j in link_pairs(cents[t], cents[t + 1]):
            edges.append((off[t] + i, off[t + 1] + j))
    return nodes, edges

print('ok')

## 5 — Yerel metrik (resmi edge Jaccard'a yakın)

Node'lar 7 µm ile GT'ye eşleştirilir, sonra kenarlar karşılaştırılır. **Seyrek GT:** iki
ucu da GT'ye eşleşmeyen tahmin kenarı yok sayılır. Bu, ayarları LB'ye submit yakmadan
ölçmemizi sağlayan pusula — eksikliğimiz buydu.

In [ ]:
def eval_vs_gt(nodes, edges, gt_ndf, gt_edges):
    pn = pd.DataFrame(nodes, columns=['node_id', 't', 'z', 'y', 'x'])
    gmap = {}
    for t, g in gt_ndf.groupby('t'):
        p = pn[pn.t == int(t)]
        if len(p) == 0 or len(g) == 0:
            continue
        D = cdist(p[['z', 'y', 'x']].values * S, g[['z', 'y', 'x']].values * S)
        cost = np.where(D <= MATCH_UM, D, 1e6)
        r, c = linear_sum_assignment(cost)
        pid = p['node_id'].values; gid = g['id'].values
        for i, j in zip(r, c):
            if D[i, j] <= MATCH_UM:
                gmap[int(pid[i])] = int(gid[j])
    gtset = set((int(u), int(v)) for u, v in gt_edges)
    TP = FP = 0; cov = set()
    for u, v in edges:
        gu, gv = gmap.get(u), gmap.get(v)
        if gu is None or gv is None:
            continue                       # etiketsiz -> yoksay
        if (gu, gv) in gtset:
            TP += 1; cov.add((gu, gv))
        else:
            FP += 1
    FN = len(gtset) - len(cov)
    return dict(jaccard=round(TP / max(TP + FP + FN, 1), 4), TP=TP, FP=FP, FN=FN,
                node_recall=round(len(set(gmap.values())) / max(len(gt_ndf), 1), 4),
                pred_nodes=len(nodes))

print('ok')

### 5a — Sanity: GT → GT skoru **1.0** olmalı (yalnızca DEV)

In [ ]:
if DEV:
    g_ndf, g_edges = load_geff(TRAIN / (test_names[0] + '.geff'))
    gt_nodes = [(int(r.id), int(r.t), float(r.z), float(r.y), float(r.x)) for r in g_ndf.itertuples()]
    chk = eval_vs_gt(gt_nodes, [(int(u), int(v)) for u, v in g_edges], g_ndf, g_edges)
    print('GT->GT:', chk)
    assert chk['jaccard'] > 0.999, 'SANITY FAIL — metrik implementasyonu hatali!'
    print('>> Metrik dogrulandi.')
else:
    print('GERCEK RERUN -> sanity atlandi (GT yok)')

## 6 — Baseline'ı değerlendir (test isimli 4 dataset, train GT ile)

Bu 4 film hem train hem test'te → GT'leri elimizde. Fit edilen parametre yok → **yanlı değil**,
gerçek skor tahmini. Resmi metrik **micro-average** (TP/FP/FN videolar arası havuzlanır).

In [ ]:
if DEV:
    res = []
    for nm in test_names:
        gp = TRAIN / (nm + '.geff')
        if not gp.exists():
            continue
        t0 = time.time()
        nodes, edges = track_dataset(open_image(TEST / (nm + '.zarr')))
        gn, ge = load_geff(gp)
        r = eval_vs_gt(nodes, edges, gn, ge)
        r.update(dataset=nm, sec=round(time.time() - t0, 1), gt_nodes=len(gn))
        res.append(r); print(nm, r)

    df = pd.DataFrame(res)
    TP, FP, FN = int(df.TP.sum()), int(df.FP.sum()), int(df.FN.sum())
    print('\n=== OZET ===')
    print(df[['dataset', 'jaccard', 'node_recall', 'TP', 'FP', 'FN', 'pred_nodes', 'gt_nodes', 'sec']].to_string(index=False))
    print(f'\nMICRO Edge Jaccard (RESMI): {TP/max(TP+FP+FN,1):.4f}   [TP={TP} FP={FP} FN={FN}]')
    print('\nFN dagilimi (kayip nerede?):')
    for r in df.sort_values('FN', ascending=False).itertuples():
        print(f'  {r.dataset}: FN={r.FN:4d} ({100*r.FN/max(FN,1):4.1f}%)  recall={r.node_recall}')
else:
    print('GERCEK RERUN -> eval atlandi (sadece submission uretilir)')

## 7 — Gönderim: `test/` dinamik gez, `submission.csv` yaz

Satır satır yaz (gizli test büyük olabilir, RAM'de tutma). Bir dataset patlarsa yer tutucu
node koy, koşuyu öldürme — her test dataset'i submission'da yer almalı.

In [ ]:
import csv
gid = tot_n = tot_e = 0; failed = []; t_start = time.time()
with open(OUT_CSV, 'w', newline='') as fh:
    w = csv.writer(fh)
    w.writerow(['id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id'])
    for k, nm in enumerate(test_names, 1):
        t0 = time.time()
        try:
            nodes, edges = track_dataset(open_image(TEST / (nm + '.zarr')))
        except Exception as e:
            print(f'  [HATA] {nm}: {type(e).__name__}: {e}'); nodes, edges = [], []; failed.append(nm)
        for nid, t, z, y, x in nodes:
            w.writerow([gid, nm, 'node', nid, t, z, y, x, -1, -1]); gid += 1
        for u, v in edges:
            w.writerow([gid, nm, 'edge', -1, -1, -1, -1, -1, u, v]); gid += 1
        if not nodes:                                    # her dataset yer almali
            w.writerow([gid, nm, 'node', 1, 0, 0, 0, 0, -1, -1]); gid += 1
            print(f'  [uyari] {nm}: tespit yok -> yer tutucu')
        tot_n += len(nodes); tot_e += len(edges)
        print(f'[{k}/{len(test_names)}] {nm}: node={len(nodes)} edge={len(edges)} ({time.time()-t0:.0f}s)')
print(f'\nYAZILDI {OUT_CSV} | satir={gid:,} node={tot_n:,} edge={tot_e:,} | {(time.time()-t_start)/60:.1f} dk')
if failed:
    print('BASARISIZ (yer tutucu kondu):', failed)

## 8 — Şema doğrulama

In [ ]:
head = pd.read_csv(OUT_CSV, nrows=5)
ss = ROOT / 'sample_submission.csv'
if ss.exists():
    assert list(head.columns) == list(pd.read_csv(ss).columns), 'KOLON UYUSMAZLIGI!'
    print('kolonlar OK')
print(head.to_string(index=False))

if DEV:                                                  # agir kontrol sadece DEV
    sub = pd.read_csv(OUT_CSV)
    missing = set(test_names) - set(sub.dataset.unique())
    assert not missing, f'eksik dataset: {missing}'
    bad = 0
    for ds, g in sub.groupby('dataset'):
        nid = set(g[g.row_type == 'node'].node_id); e = g[g.row_type == 'edge']
        bad += int((~e.source_id.isin(nid)).sum() + (~e.target_id.isin(nid)).sum())
    assert bad == 0, f'gecersiz edge referansi: {bad}'
    print(f'row_type={dict(sub.row_type.value_counts())} | dataset={sub.dataset.nunique()} | edge-ref OK')
    print('>> Submission gecerli.')
else:
    print('GERCEK RERUN -> agir kontrol atlandi | satir:', sum(1 for _ in open(OUT_CSV)) - 1)
print('\nHatirlatma: Internet = OFF olmali, yoksa submit reddedilir.')

## 9 — Sonraki adımlar

Yerel micro-Jaccard artık pusula. FN dağılımı kaybın nerede olduğunu söylüyor (EDA + Erdem'in
koşusunda kaybın ~%91'i iki `6bba` dataset'inde). Etkiye göre sıralı:

1. **6bba recall'ü** — dense doku, **instance ayrımı** asıl darboğaz (yerel SNR ~1.5×).
   Watershed / çoklu-eşik hipotezi; `SIGMA`/`FOOT` taraması. Her değişikliği yerel
   micro-Jaccard ile ölç.
2. **`44b6_0b24845f` felaketi** (recall ~0.33) — bu dataset baseline'ı yeniyor; ayrı incele.
3. **Bölünme (%10)** — şu an 1-1 Hungarian. Ebeveyn-kız ~6 µm; eşleşmeyen node'lar için
   ikinci bir eşleştirme turu (bir ebeveyn → iki çocuk).
4. **Ceza** — `pred_nodes` yüksek olsa da FP≈0 ve ceza zayıf; recall'dan önce buna dokunma.
5. **Runtime** — gizli test daha büyük olabilir; `sn/kare`'den bütçe çıkar.